# Cross-Study Marker Expression

Python notebook front end for the first major cross-study ON/OFF marker-expression plot. The reusable logic lives in `mge_organoid_python.cross_study_marker_expression`.

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd

REPO_ROOT = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents]
     if (candidate / "python_notebooks" / "src" / "mge_organoid_python").exists()),
    Path.cwd(),
)
SRC_DIR = REPO_ROOT / "python_notebooks" / "src"
if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from mge_organoid_python.cross_study_marker_expression import (
    default_cross_study_marker_specs,
    included_specs,
    plot_default_marker_grids,
    readiness_table,
    table_dir,
    write_setup_tables,
)

PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/nfs/turbo/umms-parent/mgeo_neuron_scrnaseq_projectfolder"))
RUN_LABEL = os.environ.get("CROSS_STUDY_MARKER_RUN_LABEL", "cross_study_marker_expression_v5")
MAX_CELLS_RAW = os.environ.get("CROSS_STUDY_MARKER_MAX_CELLS_PER_STUDY", "").strip()
MAX_CELLS_PER_STUDY = int(MAX_CELLS_RAW) if MAX_CELLS_RAW else None

print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"RUN_LABEL={RUN_LABEL}")
print(f"MAX_CELLS_PER_STUDY={MAX_CELLS_PER_STUDY}")

In [ ]:
setup_paths = write_setup_tables(PROJECT_ROOT, RUN_LABEL)
setup_paths

In [ ]:
specs = default_cross_study_marker_specs(PROJECT_ROOT)
ready = readiness_table(specs, PROJECT_ROOT, RUN_LABEL)
ready_path = table_dir(PROJECT_ROOT, RUN_LABEL) / "cross_study_marker_expression_readiness.tsv"
ready.to_csv(ready_path, sep="\t", index=False)
ready[["study_id", "include_in_first_plot", "seurat_exists", "h5ad_exists", "marker_table_exists", "python_ready", "n_cells_marker_table", "next_action"]]

In [ ]:
included_ids = [spec.study_id for spec in included_specs(specs)]
missing = ready.loc[ready["include_in_first_plot"] & ~ready["python_ready"], ["study_id", "next_action"]]
if not missing.empty:
    print("Not plotting yet; missing Python-ready marker tables:")
    print(missing.to_string(index=False))
else:
    manifest = plot_default_marker_grids(
        PROJECT_ROOT,
        RUN_LABEL,
        max_cells_per_study=MAX_CELLS_PER_STUDY,
    )
    print(manifest[["plot_token", "plot_path"]].drop_duplicates().to_string(index=False))